In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import csv
import pydeck as pdk
import geopandas as gpd
pd.set_option('display.max_columns', None)

In [ ]:
data2017 = pd.read_csv("donnees/élections/leg2017comm.csv", delimiter=",", low_memory=False)
data2017 = data2017[data2017["dep"] == "29"]
data2017["codecommune"] = data2017["codecommune"].astype(int)

data2022 = pd.read_csv("donnees/élections/leg2022comm.csv", delimiter=",", low_memory=False)
data2022 = data2022[data2022["dep"] == "29"]
data2022["codecommune"] = data2022["codecommune"].astype(int)

In [ ]:
import geopandas as gpd
import folium

# Charger les données
map_data = gpd.read_file("donnees/élections/communes-france.geojson")
map_data = map_data[map_data["dep_code"] == "29"]
map_data["com_code"] = map_data["com_code"].astype(int)

# Créer la carte centrée sur le Finistère
center = [map_data.geometry.centroid.y.mean(), map_data.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

# Ajouter les communes avec popup
for idx, row in map_data.iterrows():
    # Créer un popup avec les informations de la commune
    popup_text = f"<b>{row.get('com_name', 'Commune')}</b><br>Code: {row['com_code']}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x: {
            'fillColor': '#3388ff',
            'color': '#ffffff',
            'weight': 1,
            'fillOpacity': 0.5
        },
        highlight_function=lambda x: {
            'fillColor': '#ff7800',
            'color': '#ffffff',
            'weight': 1,
            'fillOpacity': 0.7
        },
        tooltip=row.get('com_name', f"Code: {row['com_code']}"),
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Afficher la carte
m.save("carte_finistere.html")
print("Carte interactive sauvegardée dans 'carte_finistere.html'")
m

In [ ]:
data_map2017 = map_data.merge(data2017, left_on="com_code", right_on="codecommune", how="left")
data_map2022 = map_data.merge(data2022, left_on="com_code", right_on="codecommune", how="left")

In [ ]:
import folium
import branca.colormap as cm
import matplotlib.colors as mcolors

# Variable à visualiser
color = 'pvoixFN'  # Attention, en 2017, c'était FN et non pas RN

# Créer la carte centrée sur le Finistère
center = [data_map2017.geometry.centroid.y.mean(), data_map2017.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

# Obtenir la palette 'autumn' de matplotlib
cmap_mpl = plt.cm.autumn

# Créer une palette de couleurs Folium compatible avec 'autumn'
vmin = data_map2017[color].min() * 100  # Multiplier par 100
vmax = data_map2017[color].max() * 100  # Multiplier par 100

# Extraire les couleurs de la colormap matplotlib 'autumn'
colors_autumn = [mcolors.rgb2hex(cmap_mpl(i)) for i in [0, 0.25, 0.5, 0.75, 1.0]]
colormap = cm.LinearColormap(
    colors=colors_autumn,
    vmin=vmin,
    vmax=vmax,
    caption='Pourcentage de voix FN 2017 (%)'
)

# Ajouter les communes avec coloration selon pvoixFN
for idx, row in data_map2017.iterrows():
    value = row[color]

    # Gérer les valeurs manquantes
    if pd.isna(value):
        fill_color = 'lightgrey'
        value_display = 'Non disponible'
    else:
        value_percent = value * 100  # Multiplier par 100
        fill_color = colormap(value_percent)
        value_display = f"{value_percent:.2f}%"

    # Récupérer les informations de la commune
    com_name = row.get('com_name', 'Commune')
    com_code = row.get('com_code', 'N/A')
    exprimes = row.get('exprimes', 'N/A')

    popup_text = f"""
    <b>{com_name}</b><br>
    Code: {com_code}<br>
    Vote FN 2017: {value_display}<br>
    Exprimés: {exprimes}
    """

    tooltip_text = f"{com_name[0]} - FN: {value_display}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#ffffff',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.9
        },
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Ajouter la légende
colormap.add_to(m)

# Ajouter un titre personnalisé
title_html = '''
<div style="position: fixed;
            bottom: 20px; left: 10px; width: 400px; height: 50px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:16px; font-weight: bold; padding: 10px">
Carte du Finistère - Vote Front National en 2017
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Sauvegarder et afficher
m.save("carte_finistere_fn2017.html")
print("Carte interactive créée avec coloration selon le vote FN 2017")
m

In [ ]:
color = 'pvoixRN'

center = [data_map2022.geometry.centroid.y.mean(), data_map2022.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

cmap_mpl = plt.cm.autumn

vmin = data_map2022[color].min() * 100  # Multiplier par 100
vmax = data_map2022[color].max() * 100  # Multiplier par 100

# Extraire les couleurs de la colormap matplotlib 'autumn'
colors_autumn = [mcolors.rgb2hex(cmap_mpl(i)) for i in [0, 0.25, 0.5, 0.75, 1.0]]
colormap = cm.LinearColormap(
    colors=colors_autumn,
    vmin=vmin,
    vmax=vmax,
    caption='Pourcentage de voix RN 2022 (%)'
)

# Ajouter les communes avec coloration selon pvoixFN
for idx, row in data_map2022.iterrows():
    value = row[color]

    # Gérer les valeurs manquantes
    if pd.isna(value):
        fill_color = 'lightgrey'
        value_display = 'Non disponible'
    else:
        value_percent = value * 100  # Multiplier par 100
        fill_color = colormap(value_percent)
        value_display = f"{value_percent:.2f}%"

    # Récupérer les informations de la commune
    com_name = row.get('com_name', 'Commune')
    com_code = row.get('com_code', 'N/A')
    exprimes = row.get('exprimes', 'N/A')

    popup_text = f"""
    <b>{com_name}</b><br>
    Code: {com_code}<br>
    Vote RN 2022: {value_display}<br>
    Exprimés: {exprimes}
    """

    tooltip_text = f"{com_name[0]} - RN: {value_display}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#ffffff',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.9
        },
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Ajouter la légende
colormap.add_to(m)

# Ajouter un titre personnalisé
title_html = '''
<div style="position: fixed;
            bottom: 20px; left: 10px; width: 400px; height: 50px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:16px; font-weight: bold; padding: 10px">
Carte du Finistère - Vote RN en 2022
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Sauvegarder et afficher
m.save("carte_finistere_rn2022.html")
print("Carte interactive créée avec coloration selon le vote RN 2022")
m

Création d'une variable d'évolution du vote RN

In [ ]:
data_map2022["VarRN"] = (data_map2022["pvoixRN"] - data_map2017["pvoixFN"])
data_map2022.head()

In [ ]:
color = 'VarRN'

center = [data_map2022.geometry.centroid.y.mean(), data_map2022.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

cmap_mpl = plt.cm.autumn

vmin = data_map2022[color].min() * 100  # Multiplier par 100
vmax = data_map2022[color].max() * 100  # Multiplier par 100

# Extraire les couleurs de la colormap matplotlib 'autumn'
colors_autumn = [mcolors.rgb2hex(cmap_mpl(i)) for i in [0, 0.25, 0.5, 0.75, 1.0]]
colormap = cm.LinearColormap(
    colors=colors_autumn,
    vmin=vmin,
    vmax=vmax,
    caption='Variation du vote RN entre 2017 et 2022 (%)'
)

for idx, row in data_map2022.iterrows():
    value = row[color]

    if pd.isna(value):
        fill_color = 'lightgrey'
        value_display = 'Non disponible'
    else:
        value_percent = value * 100
        fill_color = colormap(value_percent)
        value_display = f"{value_percent:.2f}%"

    com_name = row.get('com_name', 'Commune')
    com_code = row.get('com_code', 'N/A')
    exprimes = row.get('exprimes', 'N/A')

    popup_text = f"""
    <b>{com_name}</b><br>
    Code: {com_code}<br>
    Variation du vote RN 2017 - 2022: {value_display}<br>
    Exprimés: {exprimes}
    """

    tooltip_text = f"{com_name[0]} - Variation du voteRN: {value_display}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#ffffff',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.9
        },
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

colormap.add_to(m)

title_html = '''
<div style="position: fixed;
            bottom: 20px; left: 10px; width: 500px; height: 50px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:16px; font-weight: bold; padding: 10px">
Carte du Finistère - Variation du vote RN entre 2017 et 2022
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

m.save("carte_finistere_rn2022.html")
print("Carte interactive créée avec coloration selon la variation du vote RN entre 2017 et 2022")
m